In [ ]:
#Import key libraries and Modules
from matplotlib import ticker
import pandas as pd
import numpy as np
# Import datetime modules that comes pre-installed in Python
# datetime offers classes that work with date 7 time information
import datetime as dt
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import os
import webbrowser

# Initialize an empty list to store the company data dictionaries
companies_list = []


   System matched your input to: 'META' (META)
   [NOTICE] Error connecting to search index. Let's try that entry again.
Traceback (most recent call last):
  File "c:\Users\user\.vscode\extensions\ms-python.python-2026.4.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
  File "<string>", line 61, in <module>
AttributeError: 'NoneType' object has no attribute 'strip'



In [ ]:

# This loop keeps running until a valid integer is entered
while True:
    try:
        num_companies_input = input(
            "How many companies do you want to analyze? (Enter a number): "
        )
        num_companies = int(num_companies_input)

        # Optional check: Ensure they didn't enter 0 or a negative number
        if num_companies <= 0:
            print(
                "   [NOTICE] The number must be greater than 0. Please try again."
            )
            continue

        break  # Success! Valid number received, exit the input loop

    except ValueError:
        # This catches letters, spaces, or symbols that cannot be turned into an integer
        print(
            f"   [NOTICE] '{num_companies_input}' is not a valid number. Please enter a whole number (e.g., 3)."
        )

# A set of common English affirmative answers in lowercase.
# Since we convert inputs to lowercase, any capitalization variation (e.g., "YES", "Exactly") works automatically.
AFFIRMATIVES = {
    "y", "yes", "yeah", "yep", "yea", "yup", "ja", "ok", "okay", "sure", 
    "correct", "affirmative", "exactly", "that's right", "thats right", 
    "you got it", "right on", "absolutely", "definitely", "indeed", 
    "totally", "right", "true", "perfect", "fine", "certainly"
}

i = 0
while i < num_companies:
    while True:
        # Dynamic instruction string changes if there is a previous entry to return to
        back_instruction = " (or type 'back' to return to previous entry)" if i > 0 else ""
        
        company_input = input(
            f"\n[{i+1}/{num_companies}] Enter the company name or ticker symbol{back_instruction} (e.g., Apple (AAPL), Microsoft (MSFT), Tesla (TSLA)): "
        ).strip()

        # Handle capitalization safely by evaluating against lowercase 'back'
        if company_input.lower() == "back":
            if i > 0:
                print(f"\n   [RETURNING] Going back to modify entry #{i}...")
                companies_list.pop()  # Remove the last confirmed entry
                i -= 1                 # Decrement the outer loop index counter
                break                  # Break out of inner loop to restart the prompt for the adjusted index
            else:
                print("   [NOTICE] You are already on the first entry. Cannot go back.")
                continue

        try:
            # 1. Run the Yahoo Finance lookup
            search_result = yf.Search(company_input, max_results=1)

            # 2. Check if the result exists
            if not search_result.quotes or len(search_result.quotes) == 0:
                print(
                    f"   [NOTICE] Could not find any ticker for '{company_input}'. Please check your spelling and try again."
                )
                continue

            # FIXED: Properly extract the first match dictionary object using [0]
            first_match = search_result.quotes[0]

            # 3. Ensure a trading symbol was actually returned
            if "symbol" not in first_match:
                print(
                    f"   [NOTICE] Data found for '{company_input}' is incomplete. Please try another variant of the name or ticker symbol."
                )
                continue

            ticker_symbol = first_match["symbol"]
            
            # --- FIX: DUPLICATE CHECK ---
            # Create a quick list of all tickers already saved in your data list
            existing_tickers = [company["ticker"] for company in companies_list]
            
            if ticker_symbol in existing_tickers:
                print(
                    f"   [NOTICE] '{ticker_symbol}' has already been added to your analysis list. Duplicates might not serve a purpose. Please enter a different company."
                )
                continue  # Restarts the inner loop to ask for a new input

            exact_company_name = (
                first_match.get("longName")
                or first_match.get("shortName")
                or first_match.get("name")
                or company_input
            )

            # 4. ASK OPERATOR FOR CONFIRMATION
            print(
                f"   System matched your input to: '{exact_company_name}' ({ticker_symbol})"
            )
            
            # Capitalization fix: .lower() normalizes variations like 'YES', 'Yes', 'Y' 
            confirm = input("   Is this the company name and ticker you meant? (y/n): ").strip().lower()

            if confirm not in AFFIRMATIVES:
                print(
                    f"   [NOTICE] Got it. Let's try searching for '{company_input}' again with different keywords."
                )
                continue  # Go back to the top of the while loop and ask again

            # Package and save to list if confirmed
            company_data = {"ticker": ticker_symbol, "name": exact_company_name}
            companies_list.append(company_data)
            print(
                f"-> Successfully confirmed and added: {exact_company_name} ({ticker_symbol})"
            )
            
            i += 1  # Successfully advanced, increment company count index
            break   # Break inner loop to prompt for the next company index number

        except Exception as e:
            print(
                f"   [NOTICE] Error connecting to search index. Let's try that entry again."
            )
            continue
# Print the final compiled results outside the loop
print("\n--- Final Collected List ---")
print(companies_list)
ticker_list = [company["ticker"] for company in companies_list]
ticker_name_map = {company["ticker"]: company["name"] for company in companies_list}


   System matched your input to: 'MSFT' (MSFT)
-> Successfully confirmed and added: MSFT (MSFT)
   System matched your input to: 'Boeing' (BA)
-> Successfully confirmed and added: Boeing (BA)
   System matched your input to: 'Walmart' (WMT)
-> Successfully confirmed and added: Walmart (WMT)
   System matched your input to: 'Caterpillar' (CAT)
-> Successfully confirmed and added: Caterpillar (CAT)
   System matched your input to: 'META' (META)
-> Successfully confirmed and added: META (META)

--- Final Collected List ---
[{'ticker': 'MSFT', 'name': 'MSFT'}, {'ticker': 'BA', 'name': 'Boeing'}, {'ticker': 'WMT', 'name': 'Walmart'}, {'ticker': 'CAT', 'name': 'Caterpillar'}, {'ticker': 'META', 'name': 'META'}]


In [ ]:

#Let's prompt the user to enter the start and end dates for the stock data they want to analyze. We will use these dates to fetch the historical stock data for all the companies they entered.
while True:
    start_date_input = input(
        "\nEnter the start date for the stock data (YYYY-MM-DD): "
    ).strip()
    end_date_input = input(
        "Enter the end date for the stock data (YYYY-MM-DD): "
    ).strip()

    try:
        # Validate date format and logical order
        start_date = dt.datetime.strptime(start_date_input, "%Y-%m-%d")
        end_date = dt.datetime.strptime(end_date_input, "%Y-%m-%d")

        if start_date >= end_date:
            print(
                "   [NOTICE] Start date must be earlier than end date. Please try again."
            )
            continue

        break  # Dates are valid, exit the loop

    except ValueError:
        print(
            f"   [NOTICE] One or both of the dates you entered are not in the correct format. Please use YYYY-MM-DD and try again."
        )
print(f"\nDownloading collective database for: {ticker_list}...")


In [ ]:

# Download data for all tickers simultaneously
stock = yf.download(
    ticker_list,
    start=start_date,
    end=end_date,
    multi_level_index=True,  # Crucial for organizing multi-company data frames
    auto_adjust=False,
)

# Extract only the Adjusted Closing prices for all companies
# This yields a DataFrame where each column represents a company ticker
adj_close_df = stock["Adj Close"]


# Safety catch: If the operator only entered 1 company, force pandas to keep it as a DataFrame
if isinstance(adj_close_df, pd.Series):
    adj_close_df = adj_close_df.to_frame(name=ticker_list[0])

# --- INSERT DATA CLEANING HERE ---
# 1. Forward-fill missing prices (carries the last known price to non-trading days/holidays)
adj_close_df = adj_close_df.ffill()

# 2. Backward-fill remaining blanks (handles assets starting mid-way through your date range)
adj_close_df = adj_close_df.bfill()
# ---------------------------------
print("\n--- Master Adjusted Close Prices DataFrame (Head) ---")
print(adj_close_df.head())



--- Master Adjusted Close Prices DataFrame (Head) ---
Ticker              BA        CAT       META       MSFT        WMT
Date                                                              
2015-01-02  113.657227  69.027161  77.839149  39.681736  23.127541
2015-01-05  112.870064  65.383484  76.588982  39.316830  23.060234
2015-01-06  111.540642  64.962746  75.557060  38.739761  23.237934
2015-01-07  113.272377  65.969460  75.557060  39.231968  23.854481
2015-01-08  115.275253  66.645607  77.571259  40.386097  24.357962


In [ ]:

# Calculate returns
daily_returns_df = adj_close_df.pct_change(1) * 100
daily_returns_df = daily_returns_df.fillna(0)

print("\n--- Master Daily Returns DataFrame (Head) ---")
print(daily_returns_df.head())

# Setup the exact target folder destination path
output_folder = "d:/Alimardon/Python/Capstone Project/Multicompany analysis"
os.makedirs(output_folder, exist_ok=True)

# 1. Prepare temporary copies with renamed headers to distinguish prices from returns
price_temp_df = adj_close_df.add_prefix("Price: ")
returns_temp_df = daily_returns_df.add_prefix("Return: ")

# 2. Combine both DataFrames side-by-side along columns (axis=1)
combined_market_df = pd.concat([price_temp_df, returns_temp_df], axis=1)

# 3. Save the combined DataFrame directly into your target folder path
csv_combined_filename = os.path.join(output_folder, "combined_prices_and_returns.csv")
combined_market_df.to_csv(csv_combined_filename, index=True)

print(f"[SUCCESS] Saved Combined Market DataFrame to CSV at: {csv_combined_filename}")



--- Master Daily Returns DataFrame (Head) ---
Ticker            BA       CAT      META      MSFT       WMT
Date                                                        
2015-01-02  0.000000  0.000000  0.000000  0.000000  0.000000
2015-01-05 -0.692576 -5.278613 -1.606091 -0.919583 -0.291023
2015-01-06 -1.177834 -0.643493 -1.347350 -1.467739  0.770591
2015-01-07  1.552560  1.549679  0.000000  1.270546  2.653190
2015-01-08  1.768195  1.024940  2.665798  2.941808  2.110635
[SUCCESS] Saved Combined Market DataFrame to CSV at: d:/Alimardon/Python/Capstone Project/Multicompany analysis\combined_prices_and_returns.csv


In [ ]:

# Generate the standard descriptive statistics table
price_summary = adj_close_df.describe().round(2)
return_summary = daily_returns_df.describe().round(2)

# Calculate the maximum value for each company column
max_prices = adj_close_df.max().round(2)
max_returns = daily_returns_df.max().round(2)
# Add the max values as a new clean row labeled 'Max Value' at the bottom
price_summary.index = [f"Price {idx}" for idx in price_summary.index]
return_summary.index = [f"Return {idx}" for idx in return_summary.index]
summary_stats_df = pd.concat([price_summary, return_summary])
# Convert max metrics to frames matching the summary layout to avoid index mismatches
max_p_row = pd.DataFrame([max_prices], index=["Max Stock Price Achieved ($)"])
max_r_row = pd.DataFrame(
    [max_returns], index=["Max Daily Volatility Return (%)"]
)

# Append max rows to the main summary dataframe
summary_stats_df = pd.concat([summary_stats_df, max_p_row, max_r_row])

# Explicitly print the summary dataframe to your terminal screen
print("\n--- Summary Statistics Table ---")
print(summary_stats_df)

# Define the folder globally so it passes perfectly into all subsequent code blocks
output_folder = "d:/Alimardon/Python/Capstone Project/Multicompany analysis"
os.makedirs(output_folder, exist_ok=True)

# Combine folder path with filename for summary table
html_stats_filename = os.path.join(
    output_folder, "portfolio_statistical_summary.html"
)

# Convert the DataFrame to an HTML file inside the folder
summary_stats_df.to_html(
    html_stats_filename, classes="table table-striped table-hover"
)
print(f"\n[SUCCESS] Compiled complete summary stats to: {html_stats_filename}")




--- Summary Statistics Table ---
Ticker                                BA      CAT     META     MSFT      WMT
Price count                      2765.00  2765.00  2765.00  2765.00  2765.00
Price mean                        210.76   177.92   260.89   205.16    42.38
Price std                          76.47   113.68   173.62   142.59    23.08
Price min                          95.01    45.70    73.47    34.28    15.48
Price 25%                         151.63    96.42   141.17    68.48    23.83
Price 50%                         197.40   131.44   188.68   191.12    38.61
Price 75%                         239.59   224.64   321.64   310.98    48.37
Price max                         430.30   622.96   788.15   538.66   116.33
Return count                     2765.00  2765.00  2765.00  2765.00  2765.00
Return mean                         0.06     0.09     0.11     0.10     0.07
Return std                          2.52     1.89     2.36     1.69     1.36
Return min                        -23.85  

In [ ]:

# --- FUNCTION: PLOT LINE CHARTS ---

def plot_finance_data(df, title, output_folder, save_html=True):
    # Ensure the target directory actually exists before saving
    os.makedirs(output_folder, exist_ok=True)
    
    # Initialize the base line plot
    fig = px.line(title=title)

    # Loops over every column provided in the DataFrame
    for i in df.columns:
        fig.add_scatter(x=df.index, y=df[i], name=i)

    # Style modifications applied to the figure globally
    fig.update_traces(line=dict(width=2))
    fig.update_layout({"plot_bgcolor": "white"})

    # Clean file name generation
    base_filename = (
        title.lower().replace(" ", "_").replace("(", "").replace(")", "")
    )
    png_filename = f"{base_filename}.png"
    html_filename = f"{base_filename}.html"

    # Combine the output_folder path with the filenames securely
    png_output_path = os.path.join(output_folder, png_filename)
    html_output_path = os.path.join(output_folder, html_filename)

    # Save the high-res static PNG image in the specified folder
    fig.write_image(png_output_path, width=1200, height=600, scale=3)
    print(f" Saved chart image directly to folder: {png_output_path}")

    if save_html:
        # Save the interactive graph as an HTML file in the specified folder
        fig.write_html(html_output_path)
        print(f" Saved interactive HTML chart to folder: {html_output_path}")

    fig.show()


# Run your Plotly line comparisons
plot_finance_data(adj_close_df, "Adjusted Close Price Comparison", output_folder)
plot_finance_data(daily_returns_df, "Daily Return Comparison", output_folder)



 Saved chart image directly to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\adjusted_close_price_comparison.png
 Saved interactive HTML chart to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\adjusted_close_price_comparison.html
 Saved chart image directly to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\daily_return_comparison.png
 Saved interactive HTML chart to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\daily_return_comparison.html


In [ ]:

# --- SECTION: INTERACTIVE SMOOTH DENSITY CHART (PLOTLY) ---

# Clean the data by calculating returns and dropping missing non-trading days
clean_returns = adj_close_df.pct_change(1) * 100
clean_returns = clean_returns.dropna(how="all")

# Melt the DataFrame so Plotly can read the columns as categories
melted_returns = clean_returns.reset_index().melt(
    id_vars=clean_returns.index.name or "Date",
    var_name="Company",
    value_name="Daily Return (%)"
)

# Create an interactive, high-precision distribution plot
fig_density = px.histogram(
    melted_returns,
    x="Daily Return (%)",
    color="Company",
    title="Precise Distribution of Daily Returns",
    barmode="overlay",
    opacity=0.4,
    nbins=200  # Creates tiny bars that form a smooth distribution shape
)

# FIX: Layout configuration updated with valid axis font properties
fig_density.update_layout(
    plot_bgcolor="white",
    title=dict(font=dict(size=14)),
    xaxis=dict(
        showgrid=True, 
        gridcolor="lightgray", 
        range=[-5, 5], 
        tickfont=dict(size=10),       # Valid property for axis numbers
        title=dict(text="Daily Return (%)", font=dict(size=11))  # Valid property for label
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor="lightgray", 
        tickfont=dict(size=10),       # Valid property for axis numbers
        title=dict(text="Frequency", font=dict(size=11))         # Valid property for label
    ),
    legend=dict(font=dict(size=9))
)
fig_density.update_traces(marker=dict(line=dict(width=0.5)))

# Save as an interactive HTML file
density_html_path = os.path.join(output_folder, "precise_returns_distribution.html")
fig_density.write_html(density_html_path)
print(f" [SUCCESS] Saved interactive distribution chart to: {density_html_path}")

# Display the interactive graph
fig_density.show()



 [SUCCESS] Saved interactive distribution chart to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\precise_returns_distribution.html


In [ ]:

# --- SECTION: INTERACTIVE CORRELATION HEATMAP (PLOTLY) ---

# Calculate the correlation matrix safely
correlation_matrix = daily_returns_df.corr(numeric_only=True)

# FIX: Changed "coolwarm" to "RdBu_r" to use a valid Plotly financial color scale
fig_heatmap = px.imshow(
    correlation_matrix,
    text_auto=".2f", # Automatically overlays the correlation values rounded to 2 decimals
    color_continuous_scale="RdBu_r", # Clean Red-to-Blue divergent scaling
    title="Correlation Matrix of Daily Company Returns",
    aspect="auto" # Allows the blocks to resize intelligently when you stretch the window
)

# Shrink fonts and configure auto-scaling so it scales down beautifully
fig_heatmap.update_layout(
    plot_bgcolor="white",
    title=dict(font=dict(size=14)),
    xaxis=dict(tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
    coloraxis_showscale=False # Hides the side color bar to give the matrix more breathing room
)

# Apply font scaling directly to the inside numbers so they shrink with the boxes
fig_heatmap.update_traces(textfont=dict(size=11))

# Save as an interactive HTML file
heatmap_stats_filename = os.path.join(output_folder, "company_returns_correlation_heatmap.html")
fig_heatmap.write_html(heatmap_stats_filename)
print(f" [SUCCESS] Saved interactive correlation heatmap to: {heatmap_stats_filename}")

# Display the interactive heatmap
fig_heatmap.show()
# Let's display now pairplot between the returns of the companies to see if there are any interesting relationships. We will use seaborn for this and save it as a high-resolution image.
sns.pairplot(daily_returns_df)
pairplot_filename = os.path.join(output_folder, "returns_pairplot.png")
plt.savefig(pairplot_filename, dpi=300)
print(f" [SUCCESS] Saved returns pairplot image to: {pairplot_filename}")
plt.show()
# Let's display now pairplot between the returns of the companies to see if there are any interesting relationships. We will use seaborn for this and save it as a high-resolution image.
sns.pairplot(daily_returns_df)
pairplot_filename = os.path.join(output_folder, "returns_pairplot.png")
plt.savefig(pairplot_filename, dpi=300)
print(f" [SUCCESS] Saved returns pairplot image to: {pairplot_filename}")
plt.show()

 [SUCCESS] Saved interactive correlation heatmap to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\company_returns_correlation_heatmap.html
 [SUCCESS] Saved returns pairplot image to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\returns_pairplot.png
 [SUCCESS] Saved returns pairplot image to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\returns_pairplot.png


In [ ]:
# Let's display now pairplot between the returns of the companies to see if there are any interesting relationships. We will use seaborn for this and save it as a high-resolution image.
sns.pairplot(daily_returns_df)
pairplot_filename = os.path.join(output_folder, "returns_pairplot.png")
plt.savefig(pairplot_filename, dpi=300)
print(f" [SUCCESS] Saved returns pairplot image to: {pairplot_filename}")
plt.show()

# Function to scale stock prices to based on their initial starting price, so we can compare their relative growth over time
def scale_prices(raw_prices_df):
    scaled_prices_df = raw_prices_df.copy()
    for i in raw_prices_df.columns:  # Iterate over each company column
        initial_price = raw_prices_df[i].iloc[0]
        scaled_prices_df[i] = (raw_prices_df[i] / initial_price) 
    return scaled_prices_df
scaled_prices_df = scale_prices(adj_close_df)
#Save the scaled prices DataFrame to a CSV file in the output folder
csv_scaled_filename = os.path.join(output_folder, "scaled_stock_prices.csv")
scaled_prices_df.to_csv(csv_scaled_filename, index=False)
print(f" [SUCCESS] Saved scaled stock prices to CSV at: {csv_scaled_filename}")

print("\n--- Scaled Stock Prices DataFrame (Head) ---")
print(scaled_prices_df.head())

plot_finance_data(scaled_prices_df, "Scaled Stock Price Comparison", output_folder)


 [SUCCESS] Saved returns pairplot image to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\returns_pairplot.png
 [SUCCESS] Saved scaled stock prices to CSV at: d:/Alimardon/Python/Capstone Project/Multicompany analysis\scaled_stock_prices.csv

--- Scaled Stock Prices DataFrame (Head) ---
Ticker            BA       CAT      META      MSFT       WMT
Date                                                        
2015-01-02  1.000000  1.000000  1.000000  1.000000  1.000000
2015-01-05  0.993074  0.947214  0.983939  0.990804  0.997090
2015-01-06  0.981377  0.941119  0.970682  0.976262  1.004773
2015-01-07  0.996614  0.955703  0.970682  0.988666  1.031432
2015-01-08  1.014236  0.965498  0.996558  1.017750  1.053202
 Saved chart image directly to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\scaled_stock_price_comparison.png
 Saved interactive HTML chart to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\scaled_stock_price_comparison.html


In [ ]:
#Let's create an array that holds random portfolio weights
# Note that portfolio weights must add up to 1
import random
def generate_random_weights(num_assets):
    weights = []
    for i in range(num_assets):
        weights.append(random.random())
    
    #Let's make the num of all weight add up to
    weights = weights/np.sum(weights)  # Normalize to sum to 1
    return weights
weights = generate_random_weights(len(ticker_list))
print("\n--- Randomly Generated Portfolio Weights ---")
print(weights)
#Let's build a pie chart to visualize the distribution of the portfolio weights across the companies. We will use matplotlib for this and save it as a high-resolution image.
plt.figure(figsize=(8, 8))
plt.pie(weights, labels=ticker_list, autopct='%1.1f%%', startangle=140)
plt.title("Random Portfolio Weights Distribution")
# Save the pie chart image in the output folder
pie_chart_filename = os.path.join(output_folder, "random_portfolio_weights_distribution.png")
plt.savefig(pie_chart_filename, dpi=300)
print(f" [SUCCESS] Saved portfolio weights pie chart to: {pie_chart_filename}")
plt.show()



# %%



--- Randomly Generated Portfolio Weights ---
[0.21674866 0.20440301 0.0838241  0.29531514 0.1997091 ]
 [SUCCESS] Saved portfolio weights pie chart to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\random_portfolio_weights_distribution.png


In [ ]:
#Let's create an array that holds random portfolio weights
# Note that portfolio weights must add up to 1
import random
def generate_random_weights(num_assets):
    weights = []
    for i in range(num_assets):
        weights.append(random.random())
    
    #Let's make the num of all weight add up to
    weights = weights/np.sum(weights)  # Normalize to sum to 1
    return weights
weights = generate_random_weights(len(ticker_list))
print("\n--- Randomly Generated Portfolio Weights ---")
print(weights)




# %%



--- Randomly Generated Portfolio Weights ---
[0.50204562 0.05528102 0.05601235 0.24624036 0.14042064]


In [ ]:
#Let's create an array that holds random portfolio weights
# Note that portfolio weights must add up to 1
import random
def generate_random_weights(num_assets):
    weights = []
    for i in range(num_assets):
        weights.append(random.random())
    
    #Let's make the num of all weight add up to
    weights = weights/np.sum(weights)  # Normalize to sum to 1
    return weights
weights = generate_random_weights(len(ticker_list))
print("\n--- Randomly Generated Portfolio Weights ---")
print(weights)
#Let's build a pie chart to visualize the distribution of the portfolio weights across the companies. We will use matplotlib for this and save it as a high-resolution image.
plt.figure(figsize=(8, 8))
plt.pie(weights, labels=ticker_list, autopct='%1.1f%%', startangle=140)
plt.title("Random Portfolio Weights Distribution")
# Save the pie chart image in the output folder
pie_chart_filename = os.path.join(output_folder, "random_portfolio_weights_distribution.png")
plt.savefig(pie_chart_filename, dpi=300)
print(f" [SUCCESS] Saved portfolio weights pie chart to: {pie_chart_filename}")
plt.show()



--- Randomly Generated Portfolio Weights ---
[0.26487953 0.27141805 0.03811966 0.28735486 0.1382279 ]
 [SUCCESS] Saved portfolio weights pie chart to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\random_portfolio_weights_distribution.png


 [SUCCESS] Saved weighted portfolio returns to CSV at: d:/Alimardon/Python/Capstone Project/Multicompany analysis\weighted_portfolio_returns.csv

--- Weighted Portfolio Returns DataFrame (Head) ---
Ticker              BA         CAT  ...         WMT  Total Portfolio Return ($)
Date                                ...                                        
2015-01-02  123.654214  223.820126  ...  886.218870                 2000.000000
2015-01-05  122.797815  212.005528  ...  883.639767                 1973.482412
2015-01-06  121.351460  210.641287  ...  890.449014                 1967.127325
2015-01-07  123.235514  213.905550  ...  914.074321                 1997.780206
2015-01-08  125.414558  216.097954  ...  933.367089                 2041.759760

[5 rows x 6 columns]
 Saved chart image directly to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\total_portfolio_return_over_time.png
 Saved interactive HTML chart to folder: d:/Alimardon/Python/Capstone Project/Multico

In [ ]:
# Let's calculate the portfolio daily return by taking the percentage change of the total portfolio return over time, and then plot it to see the volatility of the portfolio.
weighted_returns_df["Portfolio Daily Return (%)"] = weighted_returns_df["Total Portfolio Return ($)"].pct_change(1) * 100
weighted_returns_df["Portfolio Daily Return (%)"] = weighted_returns_df["Portfolio Daily Return (%)"].fillna(0)
plot_finance_data(weighted_returns_df[["Portfolio Daily Return (%)"]], "Portfolio Daily Return (%) Over Time", output_folder)
# Save the portfolio daily return plot as an HTML file in the output folder
portfolio_daily_return_html_path = os.path.join(output_folder, "portfolio_daily_return.html")
fig_portfolio_daily_return = px.line(weighted_returns_df, x=weighted_returns_df.index, y="Portfolio Daily Return (%)", title="Portfolio Daily Return (%) Over Time")
fig_portfolio_daily_return.update_layout(plot_bgcolor="white")
fig_portfolio_daily_return.write_html(portfolio_daily_return_html_path)


 Saved chart image directly to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\portfolio_daily_return_%_over_time.png
 Saved interactive HTML chart to folder: d:/Alimardon/Python/Capstone Project/Multicompany analysis\portfolio_daily_return_%_over_time.html
 [SUCCESS] Saved portfolio daily return interactive chart to: d:/Alimardon/Python/Capstone Project/Multicompany analysis\portfolio_daily_return.html
